In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
from openai import OpenAI
import re


api_key = ""  # 请替换为你自己的 API key
model_name = "gpt-5"

Client = OpenAI(api_key=api_key)

INPUT_JSON = Path("")
OUTPUT_JSON = INPUT_JSON.parent / "trace_bugfree_gen12together.json"

FIELD_TITLE = "title"
FIELD_BODY  = "body"
FIELD_OUTPUT= "gen_trace"

In [2]:
EXAMPLE = """
Example

Bug Report:
-----------
Title: [BUG]: Links in cards no longer work
Body: ### Checked for duplicates?

- [X] This issue is not a duplicate

### What are the steps to reproduce this bug?

Enter a <a href="https://google.com">test</a> in a field's html.
Open card, click on this link.

### Expected behaviour

Link opens

### Actual behaviour

Nothing happens

-----------

Buggy UI Info Interaction Trace:
--------------------------
*Initial Page

*Current Screen Information:  #Current Activity: .DeckPicker.  # UI Information:click has the following group(s):1#.toolbar:[{'ImageView': 'More options'}, {'Button': 'Sync (log in)'}, {'ImageButton': 'Open drawer'}, 'AnkiDroid'];2#.[{'ImageButton': 'com.ichi2.anki:id/fab_main'}];Other Widgets with Text in This Page has the following group(s):1#.Collection is empty;2#.Start adding cards
using the + icon.;.
'action': 'click', 'feature': 'com.ichi2.anki:id/fab_main'

*Current Screen Information:  #Current Activity: .DeckPicker.  # UI Information:click has the following group(s):1#.toolbar:[{'ImageView': 'More options'}, {'Button': 'Sync (log in)'}, {'ImageButton': 'Open drawer'}, 'AnkiDroid'];2#.fabBGLayout:[{'View': 'com.ichi2.anki:id/fabBGLayout'}];3#.add_shared_layout:[{'ImageButton': 'com.ichi2.anki:id/add_shared_action'}, 'Get shared decks'];4#.add_filtered_deck_layout:[{'ImageButton': 'com.ichi2.anki:id/add_filtered_deck_action'}, 'Create filtered deck'];5#.add_deck_layout:[{'ImageButton': 'com.ichi2.anki:id/add_deck_action'}, 'Create deck'];6#.[{'ImageButton': 'com.ichi2.anki:id/fab_main'}, 'Add'];Other Widgets with Text in This Page has the following group(s):1#.Collection is empty;2#.Start adding cards
using the + icon.;.
'action': 'click', 'feature': 'Add'

*Current Screen Information:  #Current Activity: .NoteEditor.  # UI Information:set_text has the following group(s):1#.constraint_layout:[{'EditText': 'com.ichi2.anki:id/id_note_editText'}, {'ImageButton': 'com.ichi2.anki:id/id_expand_button'}, {'ImageButton': 'Attach multimedia content to the Front field'}, {'ImageButton': 'Make field Front sticky'}, 'Front'];2#.constraint_layout:[{'EditText': 'com.ichi2.anki:id/id_note_editText'}, {'ImageButton': 'com.ichi2.anki:id/id_expand_button'}, {'ImageButton': 'Attach multimedia content to the Back field'}, {'ImageButton': 'Make field Back sticky'}, 'Back'];click has the following group(s):1#.key_pos_header_voice:[{'FrameLayout': 'Voice input'}];2#.['Close features menu'];3#.['Search'];4#.['Sticker Keyboard'];5#.['GIF Keyboard'];6#.['Translate'];7#.['More features'];8#.key_pos_0_0:[{'FrameLayout': 'Q'}, '1'];9#.key_pos_0_1:[{'FrameLayout': 'W'}, '2'];10#.key_pos_0_2:[{'FrameLayout': 'E'}, '3'];11#.key_pos_0_3:[{'FrameLayout': 'R'}, '4'];12#.key_pos_0_4:[{'FrameLayout': 'T'}, '5'];13#.key_pos_0_5:[{'FrameLayout': 'Y'}, '6'];14#.key_pos_0_6:[{'FrameLayout': 'U'}, '7'];15#.key_pos_0_7:[{'FrameLayout': 'I'}, '8'];16#.key_pos_0_8:[{'FrameLayout': 'O'}, '9'];17#.key_pos_0_9:[{'FrameLayout': 'P'}, '0'];18#.['A'];19#.key_pos_1_1:[{'FrameLayout': 'S'}];20#.key_pos_1_2:[{'FrameLayout': 'D'}];21#.key_pos_1_3:[{'FrameLayout': 'F'}];22#.key_pos_1_4:[{'FrameLayout': 'G'}];23#.key_pos_1_5:[{'FrameLayout': 'H'}];24#.key_pos_1_6:[{'FrameLayout': 'J'}];25#.key_pos_1_7:[{'FrameLayout': 'K'}];26#.['L'];27#.key_pos_shift:[{'FrameLayout': 'Shift'}];28#.key_pos_2_1:[{'FrameLayout': 'Z'}];29#.key_pos_2_2:[{'FrameLayout': 'X'}];30#.key_pos_2_3:[{'FrameLayout': 'C'}];31#.key_pos_2_4:[{'FrameLayout': 'V'}];32#.key_pos_2_5:[{'FrameLayout': 'B'}];33#.key_pos_2_6:[{'FrameLayout': 'N'}];34#.key_pos_2_7:[{'FrameLayout': 'M'}];35#.key_pos_del:[{'FrameLayout': 'Delete'}];36#.key_pos_switch_to_symbol:[{'FrameLayout': 'Symbol keyboard'}];37#.key_pos_bottom_symbol_1:[{'FrameLayout': ','}];38#.key_pos_switch_to_next_language:[{'FrameLayout': 'Emoji button'}];39#.['Space'];40#.key_pos_bottom_symbol_2:[{'FrameLayout': '.'}];41#.['Enter'];42#.toolbar:[{'ImageView': 'More options'}, {'Button': 'Preview'}, {'Button': 'Save'}, {'ImageButton': 'Navigate up'}, 'Add'];43#.CardEditorTagButton:['Tags: '];44#.CardEditorCardsButton:['Cards: Card 1'];45#.editor_toolbar:[{'ImageButton': 'Create Toolbar Item'}, {'ImageButton': 'Insert MathJax Equation'}, {'ImageButton': 'Change Font Size'}, {'ImageButton': 'Insert Heading'}, {'ImageButton': 'Insert Horizontal Line'}, {'ImageButton': 'Format as Underline'}, {'ImageButton': 'Format as Italic'}, {'ImageButton': 'Format as Bold'}];spinner has the following group(s):1#.note_deck_spinner:[{'Spinner': 'com.ichi2.anki:id/note_deck_spinner'}, 'Default'];2#.note_type_spinner:[{'Spinner': 'com.ichi2.anki:id/note_type_spinner'}, 'Basic'];Other Widgets with Text in This Page has the following group(s):1#.Type:;2#.Deck:;.
'action': 'set_text', 'feature': 'Front', 'input_text': '<a href="https://google.com">test</a>'

*Current Screen Information:  #Current Activity: .NoteEditor.  # UI Information:set_text has the following group(s):1#.constraint_layout:['<a href="https://google.com">test</a>', {'EditText': 'com.ichi2.anki:id/id_note_editText'}, {'ImageButton': 'com.ichi2.anki:id/id_expand_button'}, {'ImageButton': 'Attach multimedia content to the Front field'}, {'ImageButton': 'Make field Front sticky'}, 'Front'];2#.constraint_layout:[{'EditText': 'com.ichi2.anki:id/id_note_editText'}, {'ImageButton': 'com.ichi2.anki:id/id_expand_button'}, {'ImageButton': 'Attach multimedia content to the Back field'}, {'ImageButton': 'Make field Back sticky'}, 'Back'];click has the following group(s):1#.['a'];2#.['to'];3#.['I'];4#.key_pos_header_voice:[{'FrameLayout': 'Voice input'}];5#.['Open features menu'];6#.key_pos_0_0:[{'FrameLayout': 'q'}, '1'];7#.key_pos_0_1:[{'FrameLayout': 'w'}, '2'];8#.key_pos_0_2:[{'FrameLayout': 'e'}, '3'];9#.key_pos_0_3:[{'FrameLayout': 'r'}, '4'];10#.key_pos_0_4:[{'FrameLayout': 't'}, '5'];11#.key_pos_0_5:[{'FrameLayout': 'y'}, '6'];12#.key_pos_0_6:[{'FrameLayout': 'u'}, '7'];13#.key_pos_0_7:[{'FrameLayout': 'i'}, '8'];14#.key_pos_0_8:[{'FrameLayout': 'o'}, '9'];15#.key_pos_0_9:[{'FrameLayout': 'p'}, '0'];16#.['a'];17#.key_pos_1_1:[{'FrameLayout': 's'}];18#.key_pos_1_2:[{'FrameLayout': 'd'}];19#.key_pos_1_3:[{'FrameLayout': 'f'}];20#.key_pos_1_4:[{'FrameLayout': 'g'}];21#.key_pos_1_5:[{'FrameLayout': 'h'}];22#.key_pos_1_6:[{'FrameLayout': 'j'}];23#.key_pos_1_7:[{'FrameLayout': 'k'}];24#.['l'];25#.key_pos_shift:[{'FrameLayout': 'Shift'}];26#.key_pos_2_1:[{'FrameLayout': 'z'}];27#.key_pos_2_2:[{'FrameLayout': 'x'}];28#.key_pos_2_3:[{'FrameLayout': 'c'}];29#.key_pos_2_4:[{'FrameLayout': 'v'}];30#.key_pos_2_5:[{'FrameLayout': 'b'}];31#.key_pos_2_6:[{'FrameLayout': 'n'}];32#.key_pos_2_7:[{'FrameLayout': 'm'}];33#.key_pos_del:[{'FrameLayout': 'Delete'}];34#.key_pos_switch_to_symbol:[{'FrameLayout': 'Symbol keyboard'}];35#.key_pos_bottom_symbol_1:[{'FrameLayout': ','}];36#.key_pos_switch_to_next_language:[{'FrameLayout': 'Emoji button'}];37#.['Space'];38#.key_pos_bottom_symbol_2:[{'FrameLayout': '.'}];39#.['Enter'];40#.toolbar:[{'ImageView': 'More options'}, {'Button': 'Preview'}, {'Button': 'Save'}, {'ImageButton': 'Navigate up'}, 'Add'];41#.CardEditorTagButton:['Tags: '];42#.CardEditorCardsButton:['Cards: Card 1'];43#.editor_toolbar:[{'ImageButton': 'Create Toolbar Item'}, {'ImageButton': 'Insert MathJax Equation'}, {'ImageButton': 'Change Font Size'}, {'ImageButton': 'Insert Heading'}, {'ImageButton': 'Insert Horizontal Line'}, {'ImageButton': 'Format as Underline'}, {'ImageButton': 'Format as Italic'}, {'ImageButton': 'Format as Bold'}];spinner has the following group(s):1#.note_deck_spinner:[{'Spinner': 'com.ichi2.anki:id/note_deck_spinner'}, 'Default'];2#.note_type_spinner:[{'Spinner': 'com.ichi2.anki:id/note_type_spinner'}, 'Basic'];Other Widgets with Text in This Page has the following group(s):1#.Type:;2#.Deck:;.
'action': 'click', 'feature': 'Save'

*Current Screen Information:  #Current Activity: .NoteEditor.  # UI Information:set_text has the following group(s):1#.constraint_layout:[{'EditText': 'com.ichi2.anki:id/id_note_editText'}, {'ImageButton': 'com.ichi2.anki:id/id_expand_button'}, {'ImageButton': 'Attach multimedia content to the Front field'}, {'ImageButton': 'Make field Front sticky'}, 'Front'];2#.constraint_layout:[{'EditText': 'com.ichi2.anki:id/id_note_editText'}, {'ImageButton': 'com.ichi2.anki:id/id_expand_button'}, {'ImageButton': 'Attach multimedia content to the Back field'}, {'ImageButton': 'Make field Back sticky'}, 'Back'];click has the following group(s):1#.toolbar:[{'ImageView': 'More options'}, {'Button': 'Preview'}, {'Button': 'Save'}, {'ImageButton': 'Navigate up'}, 'Add'];2#.CardEditorTagButton:['Tags: '];3#.CardEditorCardsButton:['Cards: Card 1'];4#.editor_toolbar:[{'ImageButton': 'Create Toolbar Item'}, {'ImageButton': 'Insert MathJax Equation'}, {'ImageButton': 'Change Font Size'}, {'ImageButton': 'Insert Heading'}, {'ImageButton': 'Insert Horizontal Line'}, {'ImageButton': 'Format as Underline'}, {'ImageButton': 'Format as Italic'}, {'ImageButton': 'Format as Bold'}];spinner has the following group(s):1#.note_deck_spinner:[{'Spinner': 'com.ichi2.anki:id/note_deck_spinner'}, 'Default'];2#.note_type_spinner:[{'Spinner': 'com.ichi2.anki:id/note_type_spinner'}, 'Basic'];Other Widgets with Text in This Page has the following group(s):1#.Type:;2#.Deck:;Toast message on the page: Card saved;.
'action': 'click', 'feature': 'Navigate up'

*Current Screen Information:  #Current Activity: .DeckPicker.  # UI Information:click has the following group(s):1#.toolbar:[{'ImageView': 'More options'}, {'Button': 'Sync (log in)'}, {'ImageButton': 'Open drawer'}, 'AnkiDroid', '1 card due'];2#.['com.ichi2.anki:id/DeckPickerHoriz'];3#.counts_layout:[{'LinearLayout': 'Open the deck overview page containing the number of cards to see today.'}, '1', '0', '0'];4#.[{'ImageButton': 'com.ichi2.anki:id/fab_main'}];Other Widgets with Text in This Page has the following group(s):1#.Default;2#.Studied 0 cards in 0 seconds today (0s/card);.
'action': 'click', 'feature': 'Default'

*Current Screen Information:  #Current Activity: .Reviewer.  # UI Information:click has the following group(s):1#.toolbar:[{'ImageView': 'More options'}, {'Button': 'Flag card'}, {'NAF': '[705,105][837,237]'}, {'ImageButton': 'Open drawer'}];2#.qa:['test'];3#.['com.ichi2.anki:id/touch_layer'];4#.flashcard_layout_flip:[{'FrameLayout': 'com.ichi2.anki:id/flashcard_layout_flip'}, 'Show answer'];Other Widgets with Text in This Page has the following group(s):1#.1;2#.0;3#.0;4#.AnkiDroid Flashcard;.
'action': 'click', 'feature': 'test'

*Current Screen Information:  #Current Activity: .Reviewer.  # UI Information:click has the following group(s):1#.toolbar:[{'ImageView': 'More options'}, {'Button': 'Flag card'}, {'NAF': '[705,105][837,237]'}, {'ImageButton': 'Open drawer'}];2#.qa:['test'];3#.['com.ichi2.anki:id/touch_layer'];4#.flashcard_layout_flip:[{'FrameLayout': 'com.ichi2.anki:id/flashcard_layout_flip'}, 'Show answer'];Other Widgets with Text in This Page has the following group(s):1#.1;2#.0;3#.0;4#.AnkiDroid Flashcard;.
'action': 'click', 'feature': 'test'

*Current Screen Information:  #Current Activity: .Reviewer.  # UI Information:click has the following group(s):1#.toolbar:[{'ImageView': 'More options'}, {'Button': 'Flag card'}, {'NAF': '[705,105][837,237]'}, {'ImageButton': 'Open drawer'}];2#.qa:['test'];3#.['com.ichi2.anki:id/touch_layer'];4#.flashcard_layout_flip:[{'FrameLayout': 'com.ichi2.anki:id/flashcard_layout_flip'}, 'Show answer'];Other Widgets with Text in This Page has the following group(s):1#.1;2#.0;3#.0;4#.AnkiDroid Flashcard;.

--------------------------
NCF bug observed:

"The sequence shows that after creating a card with a hyperlink in the 'Front' field and saving it, the card is shown in the reviewer with the text 'test' instead of displaying the hyperlink as expected. This indicates that the hyperlink is not being rendered or displayed correctly in the card review, which is a functionality issue."

--------------------------
Fixed UI Info Interaction Trace:
--------------------------
(For Simplicity, here just show the modified part, but you need to output the complete trace)

*Current Screen Information:  #Current Activity: .DeckPicker.  # UI Information:click has the following group(s):1#.toolbar:[{'ImageView': 'More options'}, {'Button': 'Sync (log in)'}, {'ImageButton': 'Open drawer'}, 'AnkiDroid', '1 card due'];2#.['com.ichi2.anki:id/DeckPickerHoriz'];3#.counts_layout:[{'LinearLayout': 'Open the deck overview page containing the number of cards to see today.'}, '1', '0', '0'];4#.[{'ImageButton': 'com.ichi2.anki:id/fab_main'}];Other Widgets with Text in This Page has the following group(s):1#.Default;2#.Studied 0 cards in 0 seconds today (0s/card);.
'action': 'click', 'feature': 'Default'

*Current Screen Information:  #Current Activity: .Reviewer.  # UI Information:click has the following group(s):1#.toolbar:[{'ImageView': 'More options'}, {'Button': 'Flag card'}, {'NAF': '[705,105][837,237]'}, {'ImageButton': 'Open drawer'}];2#.qa:['test'];3#.['com.ichi2.anki:id/touch_layer'];4#.flashcard_layout_flip:[{'FrameLayout': 'com.ichi2.anki:id/flashcard_layout_flip'}, 'Show answer'];Other Widgets with Text in This Page has the following group(s):1#.1;2#.0;3#.0;4#.AnkiDroid Flashcard;.
'action': 'click', 'feature': 'test'

*Current Screen Information:  #Current Activity: org.chromium.chrome.browser.ChromeTabbedActivity.  # UI Information: set_text has the following group(s): 1#.toolbar_container:[{'ImageView': 'com.android.chrome:id/toolbar_shadow'}, {'ImageButton': 'com.android.chrome:id/menu_button'}, {'FrameLayout': 'com.android.chrome:id/menu_button_wrapper'}, {'ImageButton': 'com.android.chrome:id/tab_switcher_button'}, {'LinearLayout': 'com.android.chrome:id/toolbar_buttons'}, 'google.com', {'EditText': 'com.android.chrome:id/url_bar'}, {'ImageButton': 'Connection is secure. Site information'}, {'ImageButton': 'Home'}]; Other Widgets with Text in This Page has the following group(s): 1#.Web View;.

--------------------------
Note(DO NOT include this not in your response): In this case, the buggy beahviour discribed in BR is clicking the 'test' has no effect. And the correct way is click a link will open it, lead to ideal website.  

"""

INSTRUCTION = """You are an Android test engineer. Your task is to correct a given non-crashing bug observed from a given UI information interaction trace, with a given Bug Report (BR) .

A complete UI information interaction trace is made with *Initial Page and several steps of action-Screen_Information combined. You can refer to the given trace for format.
Each action should specify the activity, action type, target feature (button name, text field, etc.), and optional input text.
For Screen GUI Information, it shows most interact-able object information and some other pure text info in the page. It should obey the Android design tradition, and refelct the bug behaviour in BR. 

You need to read the Bug report (BR) to find expected behaviour and actual behaviour. Then, guided by BR, you need to modify the given buggy trace to show the expected behaviour - in other words, fix the buggy behaviour in the trace.

Use the following template:

'action': 'action_type', 'feature': 'feature_or_direction'[, 'input_text': '...']

*Current Screen Information: #Current Activity: .ActivityName. # UI Information:
click has the following group(s): 1#.<container_or_layout>:[{'<WidgetType>': '<id_or_label>'}, '<label_or_text>', ...]; 2#.[ '<label_or_text>' , ... ]; ... switch_widget has the following group(s): 1#.[ '<ON or OFF>' , '<label_or_id>' , <optional_description> ]; set_text has the following group(s): 1#.<container>:[ '<hint_or_label>' , {'EditText': '<resource_id>'}, '<Value_of_input or null>' ]; spinner has the following group(s): 1#.<container_or_toolbar>:[ {'Spinner': '<resource_id>'} , '<selected_value>' , ... ]; check_box has the following group(s): 1#.<container_or_toolbar>:[ 'status:checked' , '<label>' ,]; 2#.[ 'status:unchecked' , '<label>' ]; scrollable has the following group(s): 1#.<container>:[ '<label_or_text>', '<label_or_text>', ... ]; Other Widgets with Text in This Page has the following group(s): 1#.<text1>; 2#.<text2>; ...Toast message on the page: <toast_text_if_any>

For click, long_click or set_text action, it specifies a 'feature', that target MUST exist in the immediately previous Current Screen Information (by text/label/id). 

Keep screens realistic for Android app. It is fine to enrich pages with plausible elements for that page type (e.g., settings options, multiple list/feed items, toolbars, FABs). If no knowledge of what should exist, using neutral fillers like “test item 1/2/3”, "test post", “test comment”.

Do not use vague placeholders like “unknown button”. If a control is icon-only or unlabeled, use a plausible resource-id style identifier (e.g., id/fab, id/next, com.app:id/fab_main).
Do not add narrative or descriptive sentence in Other Widgets with Text in This Page, like "Tap on canvas did not add text;3#.No text box or keyboard shown". You need to show the bug behaviour by the information of each step in the trace, not just describe what happen.

Refer to the NCF bug reason to identify where the buggy behaviour happens. Use BR’s Expected behavior to shape the final on-screen state within the last screen info. 
Sometimes you only need to modify several screen information, but be careful to make the whole trace consistent, avoiding obvious difference in previous steps. 
Sometimes you need to modify the whole page. Sometimes you even need to modify the actions or cut the trace. Feel free to do it.
"""

# ===== prompt builder =====
def build_prompt(title: str, body: str, buggy_trace: str, bug_reason: str) -> str:
    """
    根据新的任务需求构造提示：
    - 输入：BR(title/body)、buggy trace、bug reason
    - 目标：产出 bug-free trace（同一 case 的修复版本）
    """
    br_text = f"Title: {title}\nBody:\n{body}\n"
    instruction = (
        # 参考示例仍可包含：完整示例 + 期望产物说明
        "Below is a full example for your reference, including bug report, bug reason and UI Info Interaction Traces.\n"
        f"{EXAMPLE.strip()}\n"
        "Now fix the buggy UI Info Interaction Trace for the new case below and produce a bug-free UI Info Interaction Trace only.\n\n"
        "Bug Report:\n"
        "-----------\n"
        f"{br_text}\n"
        "Bug Reason:\n"
        "----------------------------\n"
        f"{bug_reason.strip()}\n\n"
        "Buggy UI Info Interaction Trace (to be fixed):\n"
        "---------------------------------------------\n"
        f"{buggy_trace.strip()}\n\n"
        "Requirements:\n"
        "- Keep the intent of the original case.\n"
        "- Fix issues indicated by the bug reason and any inconsistencies in the buggy trace.\n"
        "- Produce a clean, self-consistent, bug-free UI Info Interaction Trace.\n"
        "- Do not include explanations; output the final trace only.\n"
    )
    return instruction


In [3]:
import re
def extract_bug_reason(judgement):
    """
    从 judgement 中提取 bug 的 reason 字段。
    - judgement 可能是 dict / str(JSON) / 其他；都尽量鲁棒处理。
    """
    if not judgement:
        return ""
    # 已是字典
    if isinstance(judgement, dict):
        return str(judgement.get("reason", "") or "")
    # 若是字符串，尝试解析为 JSON
    if isinstance(judgement, str):
        try:
            j = json.loads(judgement)
            if isinstance(j, dict):
                return str(j.get("reason", "") or "")
        except Exception:
            # 不是 JSON，直接返回原文本
            return judgement
    # 其他类型，兜底成字符串
    return str(judgement)

def ensure_action_paragraph_breaks(text: str) -> str:
    """
    如果 'action' 不在行首（而是跟在同一行后面），
    就把它拆到新段落，并在前面补 2 个换行符。
    """
    lines = text.splitlines()
    out = []
    for line in lines:
        if "'action':" in line:
            idx = line.find("'action':")
            if idx > 0:
                prefix = line[:idx].rstrip()
                action = line[idx:]
                if prefix:
                    out.append(prefix)  # 当前行前半段
                out.append("")          # 空行1（= 第1个 \n）
                out.append(action)      # 'action': 开头的新行（与上一行之间又插入一个 \n，总计两个）
            else:
                out.append(line)        # 已在行首，保持不动
        else:
            out.append(line)
    return "\n".join(out)


In [5]:
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

if not isinstance(data, list):
    raise ValueError("Top-level of input JSON must be a list.")

print(f"Total {len(data)} records. Will generate bug-free actions_content based on existing buggy traces...")

for idx, item in enumerate(data, start=1):
    if idx > 600:
        print("end first 600")
        break
    # 字段读取：title/body 仍然来自 FIELD_TITLE / FIELD_BODY
    title = item.get(FIELD_TITLE, "") or ""
    body = item.get(FIELD_BODY, "") or ""

    # 新增读取：buggy trace 与 judgement.reason
    buggy_trace = item.get("gen_trace", "") or ""
    bug_reason = extract_bug_reason(item.get("judgement"))

    print(f"[{idx}/{len(data)}] {title[:60]}")

    prompt = build_prompt(title, body, buggy_trace, bug_reason)
    resp = Client.responses.create(
        model=model_name,
        # temperature=0.7,
        instructions=INSTRUCTION,
        input=prompt,
        reasoning={"summary": "auto"},
    )

    # 收集 reasoning summaries
    reasoning_summaries = []
    for out in getattr(resp, "output", []) or []:
        if getattr(out, "type", None) == "reasoning":
            for s in getattr(out, "summary", []) or []:
                txt = getattr(s, "text", None)
                if txt:
                    reasoning_summaries.append(txt)

    # 产物：修复后的 bug-free trace
    trace_full = resp.output_text or ""

    # 小清洗：确保每个 'action' 键前有换行
    trace_full = re.sub(r"(?<!\n)'action':", r"\n'action':", trace_full)

    print("=== Reasoning summary ===")
    for rs in reasoning_summaries:
        print(rs)
    print("=== Final output (bug-free trace) ===")
    print(trace_full)

    # 保存结果：仍写回 FIELD_OUTPUT；并保留 reasoning 概要
    item["bug-free-trace"] = trace_full
    item["reasoning_summary2"] = reasoning_summaries


Total 1827 records. Will generate bug-free actions_content based on existing buggy traces...
[1/1827] [Bug]: Can't Open Privacy Policy
=== Reasoning summary ===
**Fixing UI interaction trace**

I need to produce a corrected UI info interaction trace for the Commons app. It starts with a user logged into ContributionsActivity, then they open "More options" and click "Log out," confirming their choice. The bug occurs when tapping "Privacy Policy" on the LoginActivity, as it mistakenly shows a toast reading, "Please log in to continue" instead of opening the policy. The correct behavior is to navigate to a WebView Activity displaying the privacy policy content without requiring authentication.
**Directing to Privacy Policy WebView**

The bug report states that the "Privacy Policy" should open in a WebView. I will direct users to a .WebViewActivity with a toolbar titled "Privacy Policy." The WebView will load one of the specified privacy policy URLs, keeping it plausible. I’ll use the pack